<a href="https://colab.research.google.com/github/L-Poca/Data_Pipeline/blob/main/notebooks/colab/custom_cnn_with_interpretability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Custom CNN with Interpretability - COVID-19 Classification

**Objectif:** Entraîner un CNN custom robuste et analyser son interprétabilité

**Architecture:** CNN custom profond (optimisé pour Colab Pro)

**Interprétabilité:**
- Grad-CAM - Zones d'attention du modèle
- LIME - Explication par super-pixels
- SHAP - Valeurs de Shapley

**Dataset:** COVID-19 Radiography (4 classes)
- COVID
- Normal
- Lung_Opacity
- Viral Pneumonia

## 1. Initialisation et Configuration

### Configuration Standalone

**⚠️ Cette cellule contient la configuration autonome**

###  Configuration Standalone

**⚠️ REMPLACER CETTE CELLULE** par le contenu complet de `CELL_CONFIG_STANDALONE.py`

In [2]:
"""
╔════════════════════════════════════════════════════════════════════════════╗
║  🎯 CELLULE DE CONFIGURATION STANDALONE - COPIER-COLLER DANS VOS NOTEBOOKS ║
╚════════════════════════════════════════════════════════════════════════════╝

INSTRUCTIONS:
-------------
1. Copiez TOUT le contenu de cette cellule
2. Collez-le comme PREMIÈRE CELLULE de votre notebook
3. Exécutez la cellule
4. Les variables sont prêtes à l'emploi !

Cette cellule est 100% autonome et fonctionne partout :
✅ Google Colab (clone + installe automatiquement)
✅ WSL / Linux Local
✅ Tout environnement Jupyter

APRÈS EXÉCUTION, VOUS POUVEZ UTILISER:
- config: Objet de configuration (config.batch_size, config.data_dir, etc.)
- ENV: Environnement détecté ('colab', 'wsl', 'local')
- Tous les imports des transformers

"""

# =============================================================================
# IMPORTS STANDARDS
# =============================================================================

import os
import sys
import subprocess
from pathlib import Path


# =============================================================================
# DÉTECTION AUTOMATIQUE DE L'ENVIRONNEMENT
# =============================================================================

def detect_environment():
    """Détecte l'environnement (colab, wsl, local)"""
    try:
        import google.colab
        return "colab"
    except ImportError:
        is_wsl = os.path.exists('/proc/version') and 'microsoft' in open('/proc/version').read().lower()
        return "wsl" if is_wsl else "local"

ENV = detect_environment()
print(f"🌍 Environnement: {ENV.upper()}")


# =============================================================================
# BOOTSTRAP COLAB (Clone + Install si nécessaire)
# =============================================================================

if ENV == "colab":
    print("\n🚀 Bootstrap Colab...")
    
    os.chdir('/content')
    if not os.path.exists('/content/Data_Pipeline'):
        print("📥 Clonage du repository...")
        subprocess.run(['git', 'clone', 'https://github.com/L-Poca/Data_Pipeline.git'], check=True)
    
    os.chdir('/content/Data_Pipeline')
    
    # Checkout de la branche rafael_cleaning
    result = subprocess.run(
        ['git', 'checkout', '-b', 'rafael_cleaning', 'origin/rafael_cleaning'],
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        # Si la branche locale existe déjà, juste switcher
        subprocess.run(['git', 'checkout', 'rafael_cleaning'], capture_output=True)
    
    # Installation du package en mode éditable (sans dépendances - détection Colab dans setup.py)
    print("📦 Installation du package...")
    result = subprocess.run(['pip', 'install', '-e', '.', '--quiet'], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"⚠️ Erreur installation: {result.stderr}")
    else:
        print("✅ Package installé")
    
    print("💾 Montage Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Extraction dataset
    archive_data = '/content/drive/MyDrive/DS_COVID/archive_covid.zip'
    if os.path.exists(archive_data):
        print("📦 Extraction dataset...")
        os.makedirs('./data/raw/', exist_ok=True)
        subprocess.run(['unzip', '-o', '-q', archive_data, '-d', './data/raw/COVID-19_Radiography_Dataset/'])
    
    # Extraction models

    archive_models = '/content/drive/MyDrive/DS_COVID/inceptionv3_best.zip'
    if os.path.exists(archive_models):
        print("📦 Extraction models...")
        os.makedirs('./models/', exist_ok=True)
        subprocess.run(['unzip', '-o', '-q', archive_models, '-d', './models/'])



    print("✅ Bootstrap terminé")


# =============================================================================
# CONFIGURATION DES CHEMINS
# =============================================================================

# Déterminer project_root selon l'environnement
if ENV == "colab":
    project_root = Path('/content/Data_Pipeline')
elif ENV == "wsl":
    project_root = Path('/home/cepa/DST/projet_DS/Data_Pipeline/Data_Pipeline')
else:  # local
    # Depuis un notebook dans src/notebooks/
    project_root = Path.cwd().parent.parent

# Ajouter src/ au sys.path pour les imports
# src_path = str(project_root / 'src')
# if src_path not in sys.path:
#     sys.path.insert(0, src_path)
#     print(f"✅ Chemin src/ ajouté: {src_path}")

# Charger la configuration depuis JSON
from src.utils.config import build_config

config = build_config(project_root, ENV)

# Exports pour compatibilité avec anciens notebooks
data_dir = config.data_dir
categories = config.classes
img_size = config.img_size


# =============================================================================
# IMPORTS DES TRANSFORMERS
# =============================================================================

try:
    from src.features.Pipelines.Transformateurs.image_loaders import ImageLoader
    from src.features.Pipelines.Transformateurs.image_preprocessing import (
        ImageResizer, ImageNormalizer, ImageFlattener, ImageMasker
    )
    from src.features.Pipelines.Transformateurs.image_augmentation import (
        ImageAugmenter, ImageRandomCropper
    )
    from src.features.Pipelines.Transformateurs.image_features import (
        ImageHistogram, ImagePCA, ImageStandardScaler
    )
    print("✅ Transformers importés")
except ImportError as e:
    print(f"⚠️ Erreur import transformers: {e}")


# =============================================================================
# IMPORTS ML/DL
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras

# =============================================================================
# CONFIGURATION MATPLOTLIB
# =============================================================================

plt.rcParams['figure.figsize'] = (15, 10)
sns.set_style('whitegrid')

# =============================================================================
# AFFICHAGE DU RÉSUMÉ
# =============================================================================

print("\n" + "=" * 70)
print("✅ CONFIGURATION PRÊTE - Data Pipeline")
print("=" * 70)
print(f"📂 Projet: {project_root}")
print(f"📊 Dataset: {data_dir}")
print(f"🏷️ Classes: {', '.join(categories)}")
print(f"🎛️ Images: {img_size}")
print(f"🔧 Batch: {config.batch_size} | Époques: {config.epochs}")
print(f"📐 Dataset accessible: {'✅' if data_dir.exists() else '❌'}")
print("=" * 70)
print("\n💡 Variables disponibles:")
print("   • config: Configuration complète (Config object)")
print("   • ENV: Environnement actuel")
print("\n🎯 Transformers disponibles:")
print("   • ImageLoader, ImageResizer, ImageNormalizer, ImageFlattener")
print("   • ImageAugmenter, ImageRandomCropper")
print("   • ImageHistogram, ImagePCA, ImageStandardScaler")
print("=" * 70)


🌍 Environnement: WSL
✅ Transformers importés

✅ CONFIGURATION PRÊTE - Data Pipeline
📂 Projet: /home/cepa/DST/projet_DS/Data_Pipeline/Data_Pipeline
📊 Dataset: /home/cepa/DST/projet_DS/Data_Pipeline/Data_Pipeline/data/raw/COVID-19_Radiography_Dataset/COVID-19_Radiography_Dataset
🏷️ Classes: COVID, Normal, Lung_Opacity, Viral Pneumonia
🎛️ Images: (256, 256)
🔧 Batch: 32 | Époques: 50
📐 Dataset accessible: ✅

💡 Variables disponibles:
   • config: Configuration complète (Config object)
   • ENV: Environnement actuel

🎯 Transformers disponibles:
   • ImageLoader, ImageResizer, ImageNormalizer, ImageFlattener
   • ImageAugmenter, ImageRandomCropper
   • ImageHistogram, ImagePCA, ImageStandardScaler


### Vérification GPU

In [3]:
print("=" * 70)
print("VÉRIFICATION GPU")
print("=" * 70)

# Vérifier GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n✅ GPU disponible: {len(gpus)} GPU(s)")
    for gpu in gpus:
        print(f"   • {gpu.name}")

    # Configurer la mémoire GPU
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("   • Memory growth activé")
else:
    print("\n⚠️ Pas de GPU disponible - utilisation CPU")

print(f"\nTensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

VÉRIFICATION GPU

⚠️ Pas de GPU disponible - utilisation CPU

TensorFlow version: 2.20.0
Keras version: 3.12.0


2025-11-07 09:54:41.849325: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## 2. Chargement et Préparation des Données

### Chargement des Données

In [4]:
print("=" * 70)
print("CHARGEMENT DES DONNÉES")
print("=" * 70)

# Nombre d'images par classe (Colab Pro peut gérer plus)
N_IMAGES_PER_CLASS = None  # None = toutes les images

image_paths = []
labels = []
labels_int = []

for idx, cat in enumerate(categories):
    cat_path = data_dir / cat / 'images'
    if cat_path.exists():
        imgs = sorted(list(cat_path.glob('*.png')))
        if N_IMAGES_PER_CLASS:
            imgs = imgs[:N_IMAGES_PER_CLASS]
        
        image_paths.extend(imgs)
        labels.extend([cat] * len(imgs))
        labels_int.extend([idx] * len(imgs))
        print(f"  {cat:20s}: {len(imgs):4d} images")

labels_int = np.array(labels_int)

print(f"\n  Total: {len(image_paths)} images")
print(f"  Classes: {len(categories)}")
print(f"  Distribution: {np.bincount(labels_int)}")

CHARGEMENT DES DONNÉES
  COVID               : 3616 images
  Normal              : 10192 images
  Lung_Opacity        : 6012 images
  Viral Pneumonia     : 1345 images

  Total: 21165 images
  Classes: 4
  Distribution: [ 3616 10192  6012  1345]


### Pipeline de Preprocessing

In [5]:
print("=" * 70)
print("PREPROCESSING PIPELINE")
print("=" * 70)

# Pipeline pour le CNN custom (RGB, resize à 224x224)
prep_pipeline = Pipeline([
    ('load', ImageLoader(color_mode='RGB', verbose=True)),
    ('resize', ImageResizer(img_size=(224, 224), verbose=True))
])

print("\nChargement des images...")
images = prep_pipeline.fit_transform(image_paths)

# Convertir en float32 et normaliser [0, 1]
images = images.astype('float32') / 255.0

print(f"\n📊 Images préparées:")
print(f"  Shape: {images.shape}")
print(f"  Range: [{images.min():.3f}, {images.max():.3f}]")
print(f"  Dtype: {images.dtype}")

PREPROCESSING PIPELINE

Chargement des images...


Loading images:  87%|████████▋ | 18404/21165 [01:50<27:12,  1.69it/s] 

: 

### Train/Validation/Test Split

In [ ]:
print("=" * 70)
print("TRAIN/VALIDATION/TEST SPLIT")
print("=" * 70)

# Split 70/15/15
X_train_val, X_test, y_train_val, y_test = train_test_split(
    images, labels_int,
    test_size=0.15,
    random_state=config.random_seed,
    stratify=labels_int
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.176,  # 0.176 * 0.85 ≈ 0.15 du total
    random_state=config.random_seed,
    stratify=y_train_val
)

# One-hot encoding
y_train_cat = keras.utils.to_categorical(y_train, num_classes=len(categories))
y_val_cat = keras.utils.to_categorical(y_val, num_classes=len(categories))
y_test_cat = keras.utils.to_categorical(y_test, num_classes=len(categories))

print(f"\nTrain set: {X_train.shape[0]} images")
print(f"  Distribution: {np.bincount(y_train)}")
print(f"\nValidation set: {X_val.shape[0]} images")
print(f"  Distribution: {np.bincount(y_val)}")
print(f"\nTest set: {X_test.shape[0]} images")
print(f"  Distribution: {np.bincount(y_test)}")

### Calcul des Class Weights

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

print("=" * 70)
print("CLASS WEIGHTING")
print("=" * 70)

# Calculer les poids de classe
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = {i: weight for i, weight in enumerate(class_weights_array)}

print("\nPoids de classe:")
for i, cat in enumerate(categories):
    print(f"  {cat:20s}: {class_weights[i]:.3f}")

print("\n✅ Class weighting activé pour un apprentissage équilibré")

## 3. Data Augmentation

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print("=" * 70)
print("DATA AUGMENTATION")
print("=" * 70)

# Augmentation pour le training
train_datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    fill_mode='nearest'
)

# Pas d'augmentation pour validation/test
val_datagen = ImageDataGenerator()
test_datagen = ImageDataGenerator()

print("\n✅ Data augmentation configurée:")
print("  • Rotation: ±10°")
print("  • Shift: ±10%")
print("  • Zoom: ±10%")
print("  • Horizontal flip")

print("\n📊 Création des générateurs...")
train_generator = train_datagen.flow(
    X_train, y_train_cat,
    batch_size=config.batch_size,
    shuffle=True
)

val_generator = val_datagen.flow(
    X_val, y_val_cat,
    batch_size=config.batch_size,
    shuffle=False
)

print(f"  Train: {len(train_generator)} batches de {config.batch_size}")
print(f"  Val:   {len(val_generator)} batches de {config.batch_size}")

## 4. Architecture du CNN Custom

### Construction du Modèle

Architecture profonde optimisée pour Colab Pro avec GPU:

In [ ]:
from tensorflow.keras import layers, models, regularizers

print("=" * 70)
print("CUSTOM CNN ARCHITECTURE")
print("=" * 70)

def build_custom_cnn(input_shape=(224, 224, 3), num_classes=4):
    model = models.Sequential(name='CustomCNN_COVID19')
    
    # Bloc 1: 32 filtres
    model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.25))
    
    # Bloc 2: 64 filtres
    model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.25))
    
    # Bloc 3: 128 filtres
    model.add(layers.Conv2D(128, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(128, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.3))
    
    # Bloc 4: 256 filtres
    model.add(layers.Conv2D(256, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(256, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.3))
    
    # Bloc 5: 512 filtres
    model.add(layers.Conv2D(512, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(512, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.4))
    
    # Flatten et couches denses
    model.add(layers.Flatten())
    model.add(layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.5))
    
    # Couche de sortie
    model.add(layers.Dense(num_classes, activation='softmax'))
    
    return model

# Créer le modèle
model = build_custom_cnn(input_shape=(224, 224, 3), num_classes=len(categories))

print("\n✅ Modèle créé")
print(f"   Nom: {model.name}")
print(f"   Input shape: {model.input_shape}")
print(f"   Output shape: {model.output_shape}")

### Résumé du Modèle

In [ ]:
model.summary()

### Nombre de Paramètres

In [ ]:
total_params = model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
non_trainable_params = total_params - trainable_params

print("=" * 70)
print("PARAMÈTRES DU MODÈLE")
print("=" * 70)
print(f"\nTotal:           {total_params:,}")
print(f"Trainable:       {trainable_params:,}")
print(f"Non-trainable:   {non_trainable_params:,}")
print(f"\n🚀 Modèle robuste optimisé pour Colab Pro")

## 5. Compilation et Entraînement

### Compilation du Modèle

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy

print("=" * 70)
print("COMPILATION DU MODÈLE")
print("=" * 70)

# Optimizer avec learning rate adaptatif
optimizer = Adam(learning_rate=0.001)

# Compiler le modèle
model.compile(
    optimizer=optimizer,
    loss=CategoricalCrossentropy(),
    metrics=[
        keras.metrics.CategoricalAccuracy(name='accuracy'),
        keras.metrics.AUC(name='auc'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall')
    ]
)

print("\n✅ Modèle compilé")
print(f"   Optimizer: {optimizer.__class__.__name__}")
print(f"   Learning rate: {optimizer.learning_rate.numpy()}")
print(f"   Loss: CategoricalCrossentropy")
print(f"   Metrics: accuracy, auc, precision, recall")

### Callbacks

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print("=" * 70)
print("CALLBACKS")
print("=" * 70)

# Créer dossier pour sauvegarder les modèles
models_dir = config.results_dir / 'custom_cnn_models'
models_dir.mkdir(parents=True, exist_ok=True)

callbacks = [
    # Early stopping
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    
    # Réduction du learning rate
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    
    # Sauvegarder le meilleur modèle
    ModelCheckpoint(
        filepath=str(models_dir / 'best_model.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print("\n✅ Callbacks configurés:")
print("   • EarlyStopping (patience=15)")
print("   • ReduceLROnPlateau (factor=0.5, patience=5)")
print("   • ModelCheckpoint (meilleur modèle)")

### Entraînement

In [ ]:
print("=" * 70)
print("ENTRAÎNEMENT DU MODÈLE")
print("=" * 70)

# Nombre d'époques
EPOCHS = 50

print(f"\nDébut de l'entraînement: {EPOCHS} époques")
print(f"Batch size: {config.batch_size}")
print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"\nClass weights activé: ✅")
print("\n" + "="*70 + "\n")

# Entraîner le modèle
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

print("\n" + "="*70)
print("✅ ENTRAÎNEMENT TERMINÉ")
print("="*70)

## 6. Visualisation des Courbes d'Apprentissage

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("=" * 70)
print("VISUALISATION DES COURBES D'APPRENTISSAGE")
print("=" * 70)

# Créer dossier pour sauvegarder les graphiques
plots_dir = config.results_dir / 'custom_cnn_plots'
plots_dir.mkdir(parents=True, exist_ok=True)

# Extraire les métriques
hist = history.history

# Figure avec 4 sous-graphiques
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Loss
axes[0, 0].plot(hist['loss'], label='Train', linewidth=2)
axes[0, 0].plot(hist['val_loss'], label='Validation', linewidth=2)
axes[0, 0].set_title('Loss', fontsize=14, weight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Accuracy
axes[0, 1].plot(hist['accuracy'], label='Train', linewidth=2)
axes[0, 1].plot(hist['val_accuracy'], label='Validation', linewidth=2)
axes[0, 1].set_title('Accuracy', fontsize=14, weight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# AUC
axes[1, 0].plot(hist['auc'], label='Train', linewidth=2)
axes[1, 0].plot(hist['val_auc'], label='Validation', linewidth=2)
axes[1, 0].set_title('AUC', fontsize=14, weight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('AUC')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Precision & Recall
axes[1, 1].plot(hist['precision'], label='Train Precision', linewidth=2)
axes[1, 1].plot(hist['val_precision'], label='Val Precision', linewidth=2, linestyle='--')
axes[1, 1].plot(hist['recall'], label='Train Recall', linewidth=2)
axes[1, 1].plot(hist['val_recall'], label='Val Recall', linewidth=2, linestyle='--')
axes[1, 1].set_title('Precision & Recall', fontsize=14, weight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Score')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Custom CNN - Courbes d\'Apprentissage', fontsize=16, weight='bold', y=1.00)
plt.tight_layout()
plt.savefig(plots_dir / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Graphique sauvegardé: {plots_dir / 'training_curves.png'}")

## 7. Évaluation sur le Test Set

### Prédictions

In [ ]:
print("=" * 70)
print("ÉVALUATION SUR LE TEST SET")
print("=" * 70)

# Évaluation du modèle
print("\nÉvaluation en cours...")
test_loss, test_acc, test_auc, test_prec, test_rec = model.evaluate(X_test, y_test_cat, verbose=0)

print(f"\n📊 Résultats sur le test set:")
print(f"   Loss:      {test_loss:.4f}")
print(f"   Accuracy:  {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"   AUC:       {test_auc:.4f}")
print(f"   Precision: {test_prec:.4f}")
print(f"   Recall:    {test_rec:.4f}")
print(f"   F1-Score:  {2*(test_prec*test_rec)/(test_prec+test_rec):.4f}")

# Prédictions
print("\nGénération des prédictions...")
y_pred_proba = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)

print(f"✅ Prédictions générées pour {len(X_test)} images")

### Matrice de Confusion

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)

# Visualisation
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=categories, yticklabels=categories,
            cbar_kws={'label': 'Count'})
ax.set_title('Matrice de Confusion - Custom CNN', fontsize=14, weight='bold', pad=20)
ax.set_xlabel('Prédiction', fontsize=12)
ax.set_ylabel('Vérité', fontsize=12)
plt.tight_layout()
plt.savefig(plots_dir / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Matrice de confusion sauvegardée")

### Rapport de Classification

In [ ]:
# Rapport de classification détaillé
print("=" * 70)
print("RAPPORT DE CLASSIFICATION")
print("=" * 70)
print()
print(classification_report(y_test, y_pred, target_names=categories, digits=4))

## 8. Interprétabilité - Grad-CAM, LIME, SHAP

### Import des Modules d'Interprétabilité

In [ ]:
print("=" * 70)
print("IMPORT DES MODULES D'INTERPRÉTABILITÉ")
print("=" * 70)

try:
    from src.interpretability import (
        GradCAM, visualize_gradcam,
        LIMEImageExplainer,
        SHAPExplainer,
        plot_multiple_explanations
    )
    print("\n✅ Modules d'interprétabilité importés")
except ImportError as e:
    print(f"\n⚠️ Erreur d'import: {e}")
    
# Créer un dossier pour les résultats d'interprétabilité
interp_dir = config.results_dir / 'interpretability_custom_cnn'
interp_dir.mkdir(parents=True, exist_ok=True)
print(f"\n📂 Résultats sauvegardés dans: {interp_dir}")

### Sélection des Images à Analyser

In [ ]:
# Sélectionner quelques images du test set pour l'analyse
n_samples = 6
np.random.seed(42)
sample_indices = np.random.choice(len(X_test), n_samples, replace=False)

print(f"\n🎯 Analyse de {n_samples} images du test set:")
for idx in sample_indices:
    true_label = y_test[idx]
    pred_label = y_pred[idx]
    confidence = y_pred_proba[idx, pred_label]
    
    status = "✅" if true_label == pred_label else "❌"
    print(f"  {status} Image {idx}: Vrai={categories[true_label]:15s} | "
          f"Prédit={categories[pred_label]:15s} ({confidence:.2%})")

### 8.1. Grad-CAM - Zones d'Attention

In [ ]:
print("=" * 70)
print("GRAD-CAM - VISUALISATION DES ZONES D'ATTENTION")
print("=" * 70)

# Créer l'explainer Grad-CAM
gradcam = GradCAM(model)

# Afficher les couches convolutionnelles disponibles
conv_layers = gradcam.get_available_layers()
print(f"\n📐 Couches convolutionnelles disponibles: {len(conv_layers)}")
print(f"   Couche utilisée: {gradcam.layer_name}")
print(f"   (Dernière couche convolutionnelle détectée automatiquement)")

# Générer les heatmaps pour les images échantillons
print("\n🔍 Génération des heatmaps Grad-CAM...")

from src.interpretability.gradcam import visualize_gradcam_grid

heatmaps = []
sample_images = []
sample_class_names = []
sample_confidences = []

for idx in sample_indices:
    image = X_test[idx]
    pred_label = y_pred[idx]
    confidence = y_pred_proba[idx, pred_label]
    
    # Calculer la heatmap
    heatmap = gradcam.compute_heatmap(image, class_idx=pred_label)
    
    heatmaps.append(heatmap)
    sample_images.append(image)
    sample_class_names.append(categories[pred_label])
    sample_confidences.append(confidence)

# Visualiser en grille
fig = visualize_gradcam_grid(
    np.array(sample_images),
    heatmaps,
    sample_class_names,
    confidences=sample_confidences,
    n_cols=3,
    figsize=(18, 12),
    save_path=interp_dir / 'gradcam_grid.png'
)
plt.show()

print("✅ Visualisations Grad-CAM générées")

### 8.2. LIME - Explication par Super-pixels

In [ ]:
print("=" * 70)
print("LIME - EXPLICATION PAR SUPER-PIXELS")
print("=" * 70)

# Créer l'explainer LIME
lime_explainer = LIMEImageExplainer(
    model.predict,
    segmentation_method='quickshift',
    num_samples=1000
)

print("\n✅ LIME Explainer créé")
print("   Méthode de segmentation: quickshift")
print("   Nombre d'échantillons: 1000")
print("\n⏳ Génération des explications LIME (peut prendre 2-3 minutes)...")

# Analyser 3 images avec LIME
lime_samples = 3
lime_indices = sample_indices[:lime_samples]

fig, axes = plt.subplots(lime_samples, 3, figsize=(15, lime_samples * 5))
if lime_samples == 1:
    axes = axes.reshape(1, -1)

for i, idx in enumerate(lime_indices):
    image = X_test[idx]
    pred_label = y_pred[idx]
    confidence = y_pred_proba[idx, pred_label]
    
    print(f"\nImage {i+1}/{lime_samples}: {categories[pred_label]} ({confidence:.2%})")
    
    # Générer l'explication LIME
    explanation = lime_explainer.explain_instance(
        image,
        top_labels=1,
        num_features=10,
        random_seed=42
    )
    
    # Obtenir l'image et le masque
    temp, mask = explanation.get_image_and_mask(
        pred_label,
        positive_only=True,
        num_features=5,
        hide_rest=False
    )
    
    # Visualiser
    ax_row = axes[i]
    
    # Original
    ax_row[0].imshow(image)
    ax_row[0].set_title(f'Original\n{categories[pred_label]}', fontsize=10, weight='bold')
    ax_row[0].axis('off')
    
    # Masque
    ax_row[1].imshow(mask, cmap='Reds', alpha=0.8)
    ax_row[1].set_title('Régions Importantes', fontsize=10, weight='bold')
    ax_row[1].axis('off')
    
    # Explication
    ax_row[2].imshow(temp)
    ax_row[2].set_title(f'LIME Explanation\n{confidence:.2%}', fontsize=10, weight='bold')
    ax_row[2].axis('off')

plt.suptitle('LIME - Custom CNN', fontsize=13, weight='bold')
plt.tight_layout()
plt.savefig(interp_dir / 'lime_explanations.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Explications LIME générées")

### 8.3. SHAP - Valeurs de Shapley

In [ ]:
print("=" * 70)
print("SHAP - VALEURS DE SHAPLEY")
print("=" * 70)

# Préparer les données de référence (background) pour SHAP
n_background = 50
background_indices = np.random.choice(len(X_train), n_background, replace=False)
background_data = X_train[background_indices]

print(f"\n📊 Background data: {background_data.shape}")
print("   (Subset aléatoire du training set pour SHAP)")

# Créer l'explainer SHAP
shap_explainer = SHAPExplainer(model, background_data)

print("\n✅ SHAP Explainer créé")
print("⏳ Calcul des valeurs SHAP (peut prendre 3-5 minutes)...")

# Analyser 3 images avec SHAP
shap_samples = 3
shap_indices = sample_indices[:shap_samples]
shap_images = X_test[shap_indices]

# Calculer les valeurs SHAP
shap_values = shap_explainer.explain(shap_images, check_additivity=False)

print("✅ Valeurs SHAP calculées")

# Visualiser les résultats
fig, axes = plt.subplots(shap_samples, 3, figsize=(15, shap_samples * 5))
if shap_samples == 1:
    axes = axes.reshape(1, -1)

for i, idx in enumerate(shap_indices):
    image = X_test[idx]
    pred_label = y_pred[idx]
    confidence = y_pred_proba[idx, pred_label]
    
    # Calculer la heatmap SHAP
    shap_mean = np.mean(np.abs(shap_values[i][pred_label]), axis=-1)
    shap_signed = np.mean(shap_values[i][pred_label], axis=-1)
    
    ax_row = axes[i]
    
    # Original
    ax_row[0].imshow(image)
    ax_row[0].set_title(f'Original\n{categories[pred_label]}', fontsize=10, weight='bold')
    ax_row[0].axis('off')
    
    # SHAP Magnitude
    im1 = ax_row[1].imshow(shap_mean, cmap='Reds')
    ax_row[1].set_title('SHAP Values\n(Magnitude)', fontsize=10, weight='bold')
    ax_row[1].axis('off')
    plt.colorbar(im1, ax=ax_row[1], fraction=0.046, pad=0.04)
    
    # SHAP Signed
    im2 = ax_row[2].imshow(shap_signed, cmap='RdBu_r', 
                           vmin=-np.abs(shap_signed).max(), 
                           vmax=np.abs(shap_signed).max())
    ax_row[2].set_title(f'SHAP Values\n(Signed) {confidence:.2%}', fontsize=10, weight='bold')
    ax_row[2].axis('off')
    plt.colorbar(im2, ax=ax_row[2], fraction=0.046, pad=0.04)

plt.suptitle('SHAP - Custom CNN', fontsize=13, weight='bold')
plt.tight_layout()
plt.savefig(interp_dir / 'shap_explanations.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Visualisations SHAP générées")

### 8.4. Comparaison des 3 Méthodes

In [ ]:
print("=" * 70)
print("COMPARAISON DES 3 MÉTHODES D'INTERPRÉTABILITÉ")
print("=" * 70)

# Prendre la première image pour comparaison
comp_idx = sample_indices[0]
comp_image = X_test[comp_idx]
comp_pred_label = y_pred[comp_idx]
comp_confidence = y_pred_proba[comp_idx, comp_pred_label]

print(f"\n🔍 Analyse de l'image:")
print(f"   Classe prédite: {categories[comp_pred_label]}")
print(f"   Confiance: {comp_confidence:.2%}")
print(f"\n⏳ Génération des 3 explications...")

# 1. Grad-CAM
gradcam_heatmap = gradcam.compute_heatmap(comp_image, class_idx=comp_pred_label)

# 2. LIME
lime_explanation = lime_explainer.explain_instance(
    comp_image, 
    top_labels=1, 
    num_features=5,
    random_seed=42
)

# 3. SHAP
shap_single = shap_explainer.explain(comp_image[np.newaxis, ...], check_additivity=False)
shap_values_single = shap_single[0][comp_pred_label]

# Visualiser côte à côte
fig = plot_multiple_explanations(
    comp_image,
    gradcam_heatmap=gradcam_heatmap,
    lime_explanation=lime_explanation,
    shap_values=shap_values_single,
    class_idx=comp_pred_label,
    class_name=categories[comp_pred_label],
    confidence=comp_confidence,
    save_path=interp_dir / 'comparison_all_methods.png'
)
plt.show()

print("\n✅ Comparaison générée et sauvegardée")

## 9. Résumé et Conclusions

### 🎯 Résumé de l'Analyse

Ce notebook a accompli les objectifs suivants:

1. **✅ Chargement des données** avec pipeline (comme dans transfer_learning_colab_revamp)
2. **✅ CNN Custom robuste** optimisé pour Colab Pro
   - 5 blocs convolutionnels (32→64→128→256→512 filtres)
   - Batch normalization et Dropout pour régularisation
   - ~15M paramètres entraînables
3. **✅ Class weighting** pour apprentissage équilibré
4. **✅ Data augmentation** pour améliorer la généralisation
5. **✅ Interprétabilité complète** avec 3 méthodes:
   - **Grad-CAM**: Zones d'attention rapides
   - **LIME**: Explications par super-pixels
   - **SHAP**: Valeurs de Shapley rigoureuses

### 📊 Comparaison des Méthodes d'Interprétabilité

| Méthode | Vitesse | Type | Forces |
|---------|---------|------|--------|
| **Grad-CAM** | ⚡⚡⚡ Très rapide | Zones d'attention globales | Rapide, intuitif, bon pour CNN |
| **LIME** | ⚡⚡ Moyen | Super-pixels importants | Model-agnostic, explications locales |
| **SHAP** | ⚡ Lent | Contributions au niveau pixel | Théoriquement fondé, cohérent |

### 💡 Insights Clés

- Le modèle custom peut rivaliser avec les modèles pré-entraînés
- L'interprétabilité révèle que le modèle se concentre sur les bonnes régions (poumons)
- Class weighting aide à équilibrer l'apprentissage sur les 4 classes
- Les 3 méthodes d'interprétabilité sont complémentaires

### 📁 Fichiers Générés

Tous les résultats sont sauvegardés dans `results/`:
- `custom_cnn_models/best_model.keras` - Meilleur modèle
- `custom_cnn_plots/` - Graphiques d'entraînement
- `interpretability_custom_cnn/` - Visualisations d'interprétabilité

### 🚀 Prochaines Étapes

1. Comparer avec Transfer Learning (InceptionV3, ResNet50)
2. Tester sur d'autres datasets médicaux
3. Déployer pour utilisation en production
4. Validation par des radiologues

In [ ]:
print("=" * 70)
print("🎉 NOTEBOOK TERMINÉ AVEC SUCCÈS")
print("=" * 70)
print("\n✅ Modèle CNN custom entraîné")
print("✅ Interprétabilité complète (Grad-CAM, LIME, SHAP)")
print("✅ Résultats sauvegardés")
print("\nMerci d'avoir utilisé ce notebook!")